# **Using *MiniLMv2-L12-MNLI/XNLI* for New Fields Creation**

# All Imports Here

In [27]:
import os
glb_pth = 'd:\\GitHub\\SaaS_Product_Analysis'
os.chdir(glb_pth)

import re
import pandas as pd
import json
import torch
from pathlib import Path
from transformers import pipeline

# Importing Datasets

In [28]:
data_pth = 'data\\cleaned'

zoom_df = pd.read_csv(f'{glb_pth}\\{data_pth}\\zoom_reviews_clean.csv')
meet_df = pd.read_csv(f'{glb_pth}\\{data_pth}\\meet_reviews_clean.csv')
teams_df = pd.read_csv(f'{glb_pth}\\{data_pth}\\teams_reviews_clean.csv')
webex_df = pd.read_csv(f'{glb_pth}\\{data_pth}\\webex_reviews_clean.csv')

# Implementing Model

## *Testing MiniLMv2-L12-MNLI/XNLI*

In [29]:
pipe = pipeline(
    'zero-shot-classification',
    model='MoritzLaurer/multilingual-MiniLMv2-L12-mnli-xnli'
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5512.00it/s]


In [30]:
sequence_to_classify = (
    "🤬🤬🤬"
)

candidate_labels = ['positive', 'negative', 'neutral']

output = pipe(sequence_to_classify, candidate_labels)

max_prob = max(output['scores'])
max_idx = output['scores'].index(max_prob)

print(f'Prediction: {output['labels'][max_idx].capitalize()}\nProb: {round(max_prob, 2)*100.0}%')
print(output)

Prediction: Positive
Prob: 62.0%
{'sequence': '🤬🤬🤬', 'labels': ['positive', 'negative', 'neutral'], 'scores': [0.6238628029823303, 0.23767931759357452, 0.13845789432525635]}


## *Sentiment Analysis*

In [31]:
def sentiment_prediction(body, threshold=0.55, margin=0.10):
    candidate_labels = [
        'positive feedback or satisfaction',
        'dissatisfaction, complaint, failure, or product problem',
        'neutral feedback without praise or complaint'
    ]

    probs = []
    labels = []

    for i, texts in enumerate(body):
        output = pipe(
            texts,
            candidate_labels,
            hypothesis_template=(
                "The customer is expressing {} about their experience "
                "with the product."
            ),
            multi_label=False
        )

        top_score = output['scores'][0]
        second_score = output['scores'][1]
        top_label = output['labels'][0]

        probs.append(top_score)

        if top_score < threshold or (top_score - second_score) < margin:
            labels.append('uncertain')

        elif top_label == 'positive feedback or satisfaction':
            labels.append('positive')

        elif top_label == 'dissatisfaction, complaint, failure, or product problem':
            labels.append('negative')

        else:
            labels.append('neutral')

        if i % 100 == 0:
            print(f'Processed {i} reviews')

    return probs, labels

In [32]:
zoom_df['sentiment_score'], zoom_df['sentiment_label'] = (
    sentiment_prediction(zoom_df['body'])
)

Processed 0 reviews
Processed 100 reviews
Processed 200 reviews
Processed 300 reviews
Processed 400 reviews
Processed 500 reviews
Processed 600 reviews
Processed 700 reviews
Processed 800 reviews
Processed 900 reviews
Processed 1000 reviews
Processed 1100 reviews
Processed 1200 reviews
Processed 1300 reviews
Processed 1400 reviews
Processed 1500 reviews
Processed 1600 reviews
Processed 1700 reviews
Processed 1800 reviews
Processed 1900 reviews
Processed 2000 reviews
Processed 2100 reviews
Processed 2200 reviews
Processed 2300 reviews
Processed 2400 reviews
Processed 2500 reviews
Processed 2600 reviews
Processed 2700 reviews
Processed 2800 reviews
Processed 2900 reviews
Processed 3000 reviews
Processed 3100 reviews
Processed 3200 reviews
Processed 3300 reviews
Processed 3400 reviews
Processed 3500 reviews
Processed 3600 reviews
Processed 3700 reviews
Processed 3800 reviews
Processed 3900 reviews
Processed 4000 reviews
Processed 4100 reviews
Processed 4200 reviews
Processed 4300 reviews


In [ ]:
teams_df['sentiment_score'], teams_df['sentiment_label'] = (
    sentiment_prediction(teams_df['body'])
)

In [ ]:
meet_df['sentiment_score'], meet_df['sentiment_label'] = (
    sentiment_prediction(meet_df['body'])
)

In [ ]:
webex_df.dropna(inplace=True)
webex_df['sentiment_score'], webex_df['sentiment_label'] = (
    sentiment_prediction(webex_df['body'])
)

# *Flagging Complaint Reviews*

In [40]:
def complaint_prediction(body, threshold=0.55, margin=0.10):
    candidate_labels = [
        'customer complaint, dissatisfaction, or product problem',
        'non-complaint feedback, praise, or neutral comment'
    ]

    probs = []
    labels = []

    for i, text in enumerate(body):

        output = pipe(
            text,
            candidate_labels,
            hypothesis_template=(
                "From a product experience perspective, "
                "this customer review represents {}."
            ),
            multi_label=False
        )

        top_score = output['scores'][0]
        second_score = output['scores'][1]
        top_label = output['labels'][0]

        probs.append(top_score)

        # weak or ambiguous prediction
        if top_score < threshold or (top_score - second_score) < margin:
            labels.append('uncertain')

        elif top_label == 'customer complaint, dissatisfaction, or product problem':
            labels.append('complaint')

        else:
            labels.append('not_complaint')

        if i % 100 == 0:
            print(f'Processed {i} reviews')

    return probs, labels

In [41]:
zoom_df['complaint_score'], zoom_df['complaint_flag'] = (
    complaint_prediction(zoom_df['body'])
)

Processed 0 reviews
Processed 100 reviews
Processed 200 reviews
Processed 300 reviews
Processed 400 reviews
Processed 500 reviews
Processed 600 reviews
Processed 700 reviews
Processed 800 reviews
Processed 900 reviews
Processed 1000 reviews
Processed 1100 reviews
Processed 1200 reviews
Processed 1300 reviews
Processed 1400 reviews
Processed 1500 reviews
Processed 1600 reviews
Processed 1700 reviews
Processed 1800 reviews
Processed 1900 reviews
Processed 2000 reviews
Processed 2100 reviews
Processed 2200 reviews
Processed 2300 reviews
Processed 2400 reviews
Processed 2500 reviews
Processed 2600 reviews
Processed 2700 reviews
Processed 2800 reviews
Processed 2900 reviews
Processed 3000 reviews
Processed 3100 reviews
Processed 3200 reviews
Processed 3300 reviews
Processed 3400 reviews
Processed 3500 reviews
Processed 3600 reviews
Processed 3700 reviews
Processed 3800 reviews
Processed 3900 reviews
Processed 4000 reviews
Processed 4100 reviews
Processed 4200 reviews
Processed 4300 reviews


In [ ]:
teams_df['complaint_score'], teams_df['complaint_flag'] = (
    complaint_prediction(teams_df['body'])
)

,Unnamed: 0,reviewId,rating,body,appVersion,timestamp,sentiment_score,sentiment_label,complaint_score,complaint_flag
0,0,0dcdf6d4-dfa0-4401-b3d4-d9dbc27c825f,1,👎👎👎👎👎,NaN,2026-08-26 05:50:54,0.471210,uncertain,0.608053,complaint
1,1,d70e1f1c-83fc-495b-8035-3eb460431f82,1,Can't login after resignation,NaN,2026-08-26 05:50:12,0.401677,uncertain,0.605169,complaint
2,2,ad750e18-d0d4-40cc-8797-26e77acdbfe4,1,Worst app,NaN,2026-08-26 05:49:41,0.781650,negative,0.658770,complaint
3,3,359d855e-2a8b-4313-8120-91dff1c872ef,1,Can't login,NaN,2026-08-26 05:49:09,0.709888,negative,0.643831,complaint
4,4,dae23cb0-c77e-4f78-89ea-72b01e4c7799,1,Can't login after resignation,NaN,2026-08-26 05:48:36,0.401677,uncertain,0.605169,complaint


In [ ]:
meet_df['complaint_score'], meet_df['complaint_flag'] = (
    complaint_prediction(meet_df['body'])
)

In [ ]:
webex_df['complaint_score'], webex_df['complaint_flag'] = (
    complaint_prediction(webex_df['body'])
)

# *Topic Classification*

In [51]:
def category_prediction(body, threshold=0.20, margin=0.05):

    candidate_labels = [
        'app performance, freezing, lag, slowness, or crashes',
        'audio, microphone, camera, or video quality',
        'login, sign-in, password, authentication, or account access',
        'interface, navigation, design, or usability',
        'meeting controls, screen sharing, or presentation',
        'internet connection, joining meetings, or connection problems',
        'pricing, billing, subscription, or payment',
        'customer service, help, or support'
    ]

    probs = []
    labels = []
    sources = []

    for i, text in enumerate(body):

        text_lower = str(text).lower()

        # -------------------------
        # Rule-based classification
        # -------------------------

        # 1. Login / Authentication
        if re.search(
            r'\b(login|log in|signin|sign in|password|authentication|'
            r'verification|account access|cannot access account|'
            r"can't access account)\b",
            text_lower
        ):
            labels.append('login / authentication')
            probs.append(None)
            sources.append('rule')
            continue

        # 2. Audio / Video Quality
        elif re.search(
            r'\b(audio|microphone|mic|sound|speaker|camera|video|'
            r'blurry|voice quality|video quality|audio quality)\b',
            text_lower
        ):
            labels.append('audio / video_quality')
            probs.append(None)
            sources.append('rule')
            continue

        # 3. Meeting / Screen Sharing
        elif re.search(
            r'\b(screen share|screen sharing|share screen|presentation|'
            r'mute|unmute|host control|meeting control|breakout room)\b',
            text_lower
        ):
            labels.append('meeting / screen_sharing')
            probs.append(None)
            sources.append('rule')
            continue

        # 4. Pricing / Subscription
        elif re.search(
            r'\b(price|pricing|expensive|subscription|billing|'
            r'payment|premium|paid plan|cost)\b',
            text_lower
        ):
            labels.append('pricing / subscription')
            probs.append(None)
            sources.append('rule')
            continue

        # 5. Customer Support
        elif re.search(
            r'\b(customer support|customer service|support team|'
            r'help desk|technical support|contacted support)\b',
            text_lower
        ):
            labels.append('customer_support')
            probs.append(None)
            sources.append('rule')
            continue

        # 6. Connectivity / Joining
        elif re.search(
            r'\b(connection|connectivity|disconnect|disconnected|'
            r'network|join meeting|joining meeting|cannot join|'
            r"can't join|connection drop)\b",
            text_lower
        ):
            labels.append('connectivity / joining')
            probs.append(None)
            sources.append('rule')
            continue

        # 7. Performance / Crashes
        elif re.search(
            r'\b(crash|crashes|crashing|freeze|freezing|lag|lagging|'
            r'slow|sluggish|stuck|performance)\b',
            text_lower
        ):
            labels.append('performance / crashes')
            probs.append(None)
            sources.append('rule')
            continue

        # 8. UI / UX
        elif re.search(
            r'\b(interface|navigation|layout|design|ui|ux|'
            r'confusing|difficult to use|hard to use|user interface)\b',
            text_lower
        ):
            labels.append('ui / ux')
            probs.append(None)
            sources.append('rule')
            continue

        # -------------------------
        # Model fallback
        # -------------------------

        output = pipe(
            text,
            candidate_labels,
            hypothesis_template=(
                "The primary product area discussed "
                "in this customer review is {}."
            ),
            multi_label=False
        )

        top_score = output['scores'][0]
        second_score = output['scores'][1]
        top_label = output['labels'][0]

        probs.append(top_score)

        # weak or ambiguous prediction
        if (
            top_score < threshold
            or (top_score - second_score) < margin
        ):
            labels.append('other')
            sources.append('other')

        elif top_label == 'app performance, freezing, lag, slowness, or crashes':
            labels.append('performance / crashes')
            sources.append('model')

        elif top_label == 'audio, microphone, camera, or video quality':
            labels.append('audio / video_quality')
            sources.append('model')

        elif top_label == 'login, sign-in, password, authentication, or account access':
            labels.append('login / authentication')
            sources.append('model')

        elif top_label == 'interface, navigation, design, or usability':
            labels.append('ui / ux')
            sources.append('model')

        elif top_label == 'meeting controls, screen sharing, or presentation':
            labels.append('meeting / screen_sharing')
            sources.append('model')

        elif top_label == 'internet connection, joining meetings, or connection problems':
            labels.append('connectivity / joining')
            sources.append('model')

        elif top_label == 'pricing, billing, subscription, or payment':
            labels.append('pricing / subscription')
            sources.append('model')

        elif top_label == 'customer service, help, or support':
            labels.append('customer_support')
            sources.append('model')

        else:
            labels.append('other')
            sources.append('other')

        if i % 100 == 0:
            print(f'Processed {i} reviews')

    return probs, labels, sources

In [53]:
zoom_df['category_score'], zoom_df['category'], zoom_df['category_source'] = (
    category_prediction(zoom_df['body'])
)

Processed 0 reviews
Processed 100 reviews
Processed 300 reviews
Processed 400 reviews
Processed 500 reviews
Processed 600 reviews
Processed 700 reviews
Processed 800 reviews
Processed 1000 reviews
Processed 1100 reviews
Processed 1200 reviews
Processed 1300 reviews
Processed 1400 reviews
Processed 1600 reviews
Processed 1700 reviews
Processed 1800 reviews
Processed 1900 reviews
Processed 2000 reviews
Processed 2100 reviews
Processed 2200 reviews
Processed 2300 reviews
Processed 2400 reviews
Processed 2500 reviews
Processed 2600 reviews
Processed 2800 reviews
Processed 2900 reviews
Processed 3000 reviews
Processed 3100 reviews
Processed 3200 reviews
Processed 3300 reviews
Processed 3400 reviews
Processed 3500 reviews
Processed 3600 reviews
Processed 3700 reviews
Processed 3800 reviews
Processed 3900 reviews
Processed 4000 reviews
Processed 4100 reviews
Processed 4200 reviews
Processed 4300 reviews
Processed 4400 reviews
Processed 4500 reviews
Processed 4600 reviews
Processed 4700 review

In [ ]:
teams_df['category_score'], teams_df['category'], teams_df['category_source'] = (
    category_prediction(teams_df['body'])
)

<StringArray>
[                   'other',   'login / authentication',
         'customer_support',   'connectivity / joining',
    'audio / video_quality',   'pricing / subscription',
    'performance / crashes',                  'ui / ux',
 'meeting / screen_sharing']
Length: 9, dtype: str

In [ ]:
meet_df['category_score'], meet_df['category'], meet_df['category_source'] = (
    category_prediction(meet_df['body'])
)

In [ ]:
webex_df['category_score'], webex_df['category'], webex_df['category_source'] = (
    category_prediction(webex_df['body'])
)